# Human in the Loop Agent

In [1]:
!uv pip install -U langchain langchain_ollama langgraph-checkpoint-sqlite

Using Python 3.12.3 environment at: /home/acm/envs/vllmvenv


Resolved 44 packages in 592ms                                        
Prepared 21 packages in 568ms                                                pydantic-core               ------------------------------ 1.89 MiB/2.16 MiB    pydantic-core               ------------------------------ 622.12 KiB/2.16 MiB  pydantic-core               ------------------------------ 417.29 KiB/2.16 MiB  pydantic-core               ------------------------------ 401.29 KiB/2.16 MiB  pydantic-core               ------------------------------ 401.29 KiB/2.16 MiB  pydantic-core               ------------------------------ 302.12 KiB/2.16 MiB  pydantic-core               ------------------------------ 254.12 KiB/2.16 MiB  pydantic-core               ------------------------------ 222.12 KiB/2.16 MiB  mpydantic-core               ------------------------------ 190.02 KiB/2.16 MiB  pydantic-core               ------------------------------ 174.02 KiB/2.16 MiB  ydantic-core               -------------------------

In [2]:
from langgraph.graph import StateGraph, END 
from typing import TypedDict, Annotated
import operator 
from uuid import uuid4

from langchain_core.messages import AnyMessage, ToolMessage, SystemMessage, HumanMessage 
from langchain_core.tools import tool

from langchain_ollama import ChatOllama
llm = "qwen3.8:27b"

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)



In [3]:
# tools 
@tool
def get_weather(city: str) -> str:
    """Get current weather conditions for a single city."""
    import requests
    resp = requests.get(f"https://wttr.in/{city}?format=3")
    return resp.text
    
@tool
def get_search_query(query: str) -> str: 
    """Searches web for a query and returns search results."""
    try: 
        results = DDGS().text(query, max_results=15)
    except Exception as e:
        return f"Error calling tool: {e}"
    if not results:
        return "No results found" 
    return "\n\n".join(
        f"Title: {r['title']}\nURl: {r['href']}\n Body: {r['body']}"
        for r in results
    )

In [4]:
"""
Instead of appending new messages, if a new message comes in with teh same id, replace it. 
"""

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign any messages an id if it doesnt exist already 
    for message in right: 
        if not message.id:
            message.id = str(uuid4())

    merged = left.copy()
    for message in right: 
        for i, existing in enumerate(left):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged

class AgentState(TypedDict): 
    messages: Annotated[list[AnyMessage], reduce_messages]

In [5]:
# major change -> before each action node is called, we interrupt. 

class Agent: 
    def __init__(self, model, tools, checkpointer, system=""): 
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_ollama)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True:"action", False: END})
        graph.add_edge("action", "llm") 
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"])
        self.tools = {t.name: t for t in tools} 
        self.model = model.bind_tools(tools) 
        
    def call_ollama(self, state:AgentState): 
        messages = state['messages'] 
        if self.system: 
            messages = [SystemMessage(content=self.system)] + messages 
        message = self.model.invoke(messages) 
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [6]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOllama(model=llm)  
abot = Agent(model, [get_weather, get_search_query], system=prompt, checkpointer=memory)

In [7]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-12T23:36:43.904667976Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11611030829, 'load_duration': 11063675562, 'prompt_eval_count': 402, 'prompt_eval_duration': 244980000, 'eval_count': 57, 'eval_duration': 296745000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d21e-bd23-733c-b856-c83f1012f525-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '51811e06-3c1e-40fc-8c93-42567962f130', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 57, 'total_tokens': 459})]}
()


In [8]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='db70668e-ac81-45ff-b5ce-37b574a4cebf'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-12T23:36:43.904667976Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11611030829, 'load_duration': 11063675562, 'prompt_eval_count': 402, 'prompt_eval_duration': 244980000, 'eval_count': 57, 'eval_duration': 296745000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d21e-bd23-733c-b856-c83f1012f525-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '51811e06-3c1e-40fc-8c93-42567962f130', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 57, 'total_tokens': 459})]}, next=('action',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1def1e-b29a

In [9]:
abot.graph.get_state(thread).next

('action',)

In [27]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '3a11727b-034f-4b49-a39f-45c26385bbca', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='Los Angeles: ☀️  +76°F\n', name='get_weather', id='01702a96-2439-48c0-b228-91d79eb68b27', tool_call_id='3a11727b-034f-4b49-a39f-45c26385bbca')]}
{'messages': [AIMessage(content="Here's the current weather in Los Angeles:\n\n☀️ **Sunny**, 76°F (about 24°C)\n\nA pleasant, sunny day in LA! Is there anything else I can help you with?", additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-12T23:46:04.676125533Z', 'done': True, 'done_reason': 'stop', 'total_duration': 744432199, 'load_duration': 1654429, 'prompt_eval_count': 724, 'prompt_eval_duration': 153361000, 'eval_count': 102, 'eval_duration': 585933000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d227-761a-740e-b1f5-f0de3a79be52-0', tool_calls=[], invalid_tool_calls=[],

In [28]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content="What's the weather in LA?", additional_kwargs={}, response_metadata={}, id='a776037b-4747-46be-9a6a-0bffeb105485'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-12T23:41:49.579224973Z', 'done': True, 'done_reason': 'stop', 'total_duration': 452680693, 'load_duration': 1293735, 'prompt_eval_count': 403, 'prompt_eval_duration': 210822000, 'eval_count': 53, 'eval_duration': 236835000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d223-92c5-702f-a1ab-17deccf5c924-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '8bfcca66-8735-46ee-81d3-6c9755310d23', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 403, 'output_tokens': 53, 'total_tokens': 456}), HumanMessage(content="What's the weather in LA?", additional_kwargs={}, response_metadata={}, id='095588bd-0a73-41c7-98f9-

In [29]:
abot.graph.get_state(thread).next

()

In [58]:
messages = [HumanMessage("What's the weather in London?")] 
thread = {"configurable": {"thread_id":"2"}}
for event in abot.graph.stream({"messages": messages}, thread):
     for v in event.values():
         print(v)
while abot.graph.get_state(thread).next:
    print(abot.graph.get_state(thread))
    _input = input("proceed? y/n: ")
    if _input != "y":
        print("aborting...")
        break
    for event in abot.graph.stream(None, thread):
     for v in event.values():
         print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-13T00:07:00.136162227Z', 'done': True, 'done_reason': 'stop', 'total_duration': 693837083, 'load_duration': 1832279, 'prompt_eval_count': 563, 'prompt_eval_duration': 231293000, 'eval_count': 48, 'eval_duration': 237343000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d23a-9e71-7379-be3b-d316979403b2-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'cc1b466c-eafa-484b-8d20-b1cbd915378a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 563, 'output_tokens': 48, 'total_tokens': 611})]}
()
StateSnapshot(values={'messages': [HumanMessage(content="What's the weather in London?", additional_kwargs={}, response_metadata={}, id='6f21f494-8051-4f91-8023-7f670038fa45'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '202

In [71]:
messages = [HumanMessage("What's the weather in LA?")]
thread = {"configurable": {"thread_id":"3"}}

for event in abot.graph.stream( {"messages": messages}, thread):
    for v in event.values():
        print(v)


{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-13T00:12:42.386070826Z', 'done': True, 'done_reason': 'stop', 'total_duration': 829292624, 'load_duration': 1454852, 'prompt_eval_count': 403, 'prompt_eval_duration': 234914000, 'eval_count': 55, 'eval_duration': 296861000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d23f-d6d3-729e-86b1-5c40676f2f0c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 403, 'output_tokens': 55, 'total_tokens': 458})]}
()


In [72]:
current_values = abot.graph.get_state(thread)
print(current_values.next)                                # ('action',)
print(current_values.values['messages'][-1].tool_calls)
print(current_values.values['messages'][-1].tool_calls[0])
print(current_values.values['messages'][-1].tool_calls[0]['id'])

('action',)
[{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}]
{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}
9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c


In [73]:
# override the tool call with a new message with the same id.
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {
        'name': 'get_weather',
        'args': {'city': 'New York City'},
        'id': _id,
        'type': 'tool_call'
        
    }
]

In [74]:
abot.graph.update_state(thread, current_values.values)

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1def6f-8f13-6304-8002-2e13f49d1d1d'}}

In [75]:
abot.graph.get_state(thread).next

('action',)

In [76]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'get_weather', 'args': {'city': 'New York City'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='New York City: ☁️  +52°F\n', name='get_weather', id='a9438b96-73d8-4721-9fd0-4fd52055f9b1', tool_call_id='9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c')]}
{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-13T00:13:03.66856226Z', 'done': True, 'done_reason': 'stop', 'total_duration': 571170092, 'load_duration': 1397461, 'prompt_eval_count': 460, 'prompt_eval_duration': 160545000, 'eval_count': 61, 'eval_duration': 406312000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d240-2af8-76a3-a458-bc6cbe27b82b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '7e4dce43-7763-4a70-9bfe-60b5165f0a04', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [ ]:
# build a state list to replay a previous state.
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print("--")
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content="What's the weather in LA?", additional_kwargs={}, response_metadata={}, id='858e0c3d-29b3-4e48-8797-8a25339386d5'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-13T00:12:42.386070826Z', 'done': True, 'done_reason': 'stop', 'total_duration': 829292624, 'load_duration': 1454852, 'prompt_eval_count': 403, 'prompt_eval_duration': 234914000, 'eval_count': 55, 'eval_duration': 296861000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d23f-d6d3-729e-86b1-5c40676f2f0c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York City'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 403, 'output_tokens': 55, 'total_tokens': 458}), ToolMessage(content='New York City: ☁️  +52°F\n', name='get_weather', id='a9438b96-73d8-4721-9fd0-4fd52055f9b1', tool_ca

In [84]:
to_replay = states[3]

In [85]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content="What's the weather in LA?", additional_kwargs={}, response_metadata={}, id='858e0c3d-29b3-4e48-8797-8a25339386d5'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-13T00:12:42.386070826Z', 'done': True, 'done_reason': 'stop', 'total_duration': 829292624, 'load_duration': 1454852, 'prompt_eval_count': 403, 'prompt_eval_duration': 234914000, 'eval_count': 55, 'eval_duration': 296861000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d23f-d6d3-729e-86b1-5c40676f2f0c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 403, 'output_tokens': 55, 'total_tokens': 458})]}, next=('action',), config={'configurable': {'thread_id': '3', 'checkpoint_ns': '', 'checkpoint_id': '1f1def6f-1b7e-6794-8

In [86]:
for event in abot.graph.stream(None, to_replay.config):
    for v in event.items():
        print(v)

Calling: {'name': 'get_weather', 'args': {'city': 'Los Angeles'}, 'id': '9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c', 'type': 'tool_call'}
Back to the model!
('action', {'messages': [ToolMessage(content='Los Angeles: ☀️  +76°F\n', name='get_weather', id='7e34c5d7-ff04-42a6-b1d5-60fb50eac193', tool_call_id='9e53038c-1bc6-4a86-9cbe-42f38b9ddc2c')]})
('llm', {'messages': [AIMessage(content="It's currently **76°F (24°C)** and **sunny ☀️** in Los Angeles. Nice weather!", additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-13T00:16:46.775830958Z', 'done': True, 'done_reason': 'stop', 'total_duration': 445382677, 'load_duration': 1584023, 'prompt_eval_count': 458, 'prompt_eval_duration': 160666000, 'eval_count': 44, 'eval_duration': 279857000, 'logprobs': None, 'model_name': 'qwen3.8:27b', 'model_provider': 'ollama'}, id='lc_run--01a1d243-92f9-7594-8391-c1f7d32990a5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 458, 'output_tokens': 44

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [
    { 
      'name': 'get_weather',
        'args': {'city': 'New York City'}, 
        'id': _id,
        'type': 'tool_call'
    }
]